# 05 -- Websockets et temps reel

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- comprendre le protocole WebSocket vs HTTP ;
- creer un serveur WebSocket avec la bibliotheque `websockets` ;
- utiliser les WebSockets dans FastAPI et Starlette ;
- gerer connexion, message, disconnect ;
- broadcaster a plusieurs clients ;
- echanger du JSON structure ;
- gerer l'authentification (tokens) ;
- utiliser Server-Sent Events (SSE) en alternative.

## Prerequis -- ce que vous devez maitriser

- asyncio (async/await) ;
- HTTP basics ;
- JSON ;
- FastAPI est un plus (pour la section dediee).

Ce que ce notebook **ne couvrira pas** :

- Socket.IO (bibliotheque differente) ;
- WebRTC (temps reel media) ;
- le deploiement derriere un reverse proxy.

## Plan

1. WebSocket vs HTTP
2. Serveur websockets minimal
3. Client websockets
4. Echange JSON
5. Broadcast a plusieurs clients
6. FastAPI + WebSocket
7. Gestion des deconnexions
8. Authentification par token
9. Rate limiting
10. Heartbeat et ping/pong
11. Server-Sent Events (SSE)
12. Synthese
13. Exercices

---

## 1. WebSocket vs HTTP

| Aspect | HTTP | WebSocket |
|---|---|---|
| Direction | requete-reponse | bidirectionnel full-duplex |
| Connexion | ephemere | persistante |
| Overhead | headers a chaque requete | handshake puis frames legers |
| Use case | API REST | chat, notifications, temps reel |
| Port | 80/443 | 80/443 (upgrade depuis HTTP) |

**Handshake** : HTTP GET avec `Upgrade: websocket` puis la connexion devient WS.

---

## 2. Serveur websockets minimal

In [ ]:
# pip install websockets
import websockets
print('version :', websockets.__version__)

In [ ]:
import asyncio
import websockets

async def echo(ws):
    async for message in ws:
        await ws.send(f'Echo: {message}')

async def main():
    async with websockets.serve(echo, 'localhost', 8765):
        await asyncio.sleep(0)  # demo, ne lance pas
        return 'ready'

print('Serveur defini : lancer avec asyncio.run(main()) hors notebook')

---

## 3. Client websockets

In [ ]:
# Client basique
async def client_demo():
    async with websockets.connect('ws://localhost:8765') as ws:
        await ws.send('Hello')
        response = await ws.recv()
        return response

print('Client defini')

---

## 4. Echange JSON

In [ ]:
import json

async def json_handler(ws):
    async for raw in ws:
        data = json.loads(raw)
        reply = {'ok': True, 'received': data, 'type': data.get('type')}
        await ws.send(json.dumps(reply))

print('handler JSON defini')

---

## 5. Broadcast a plusieurs clients

In [ ]:
connected: set = set()

async def broadcast_handler(ws):
    connected.add(ws)
    try:
        async for message in ws:
            # Envoie a tous les clients sauf l'emetteur
            for client in connected:
                if client is not ws:
                    await client.send(message)
    finally:
        connected.remove(ws)

print('broadcast handler defini')

---

## 6. FastAPI + WebSocket

In [ ]:
# pip install fastapi uvicorn
from fastapi import FastAPI, WebSocket, WebSocketDisconnect

api = FastAPI()

@api.websocket('/ws')
async def websocket_endpoint(ws: WebSocket):
    await ws.accept()
    try:
        while True:
            data = await ws.receive_text()
            await ws.send_text(f'Reponse : {data}')
    except WebSocketDisconnect:
        print('Client deconnecte')

print('endpoint ws defini')

In [ ]:
# Envoi JSON
@api.websocket('/ws/json')
async def ws_json(ws: WebSocket):
    await ws.accept()
    await ws.send_json({'status': 'connected'})
    while True:
        msg = await ws.receive_json()
        await ws.send_json({'echo': msg})

print('ws json defini')

---

## 7. Gestion des deconnexions

In [ ]:
from fastapi import WebSocket, WebSocketDisconnect

class ConnectionManager:
    def __init__(self) -> None:
        self.active: list[WebSocket] = []

    async def connect(self, ws: WebSocket) -> None:
        await ws.accept()
        self.active.append(ws)

    def disconnect(self, ws: WebSocket) -> None:
        if ws in self.active:
            self.active.remove(ws)

    async def broadcast(self, message: str) -> None:
        for ws in self.active:
            await ws.send_text(message)

manager = ConnectionManager()
print('manager cree')

In [ ]:
@api.websocket('/chat/{client_id}')
async def chat(ws: WebSocket, client_id: int):
    await manager.connect(ws)
    try:
        while True:
            data = await ws.receive_text()
            await manager.broadcast(f'Client #{client_id}: {data}')
    except WebSocketDisconnect:
        manager.disconnect(ws)
        await manager.broadcast(f'Client #{client_id} parti')

print('chat defini')

---

## 8. Authentification par token

In [ ]:
from fastapi import Query, WebSocket

@api.websocket('/secure')
async def secure(ws: WebSocket, token: str = Query(...)):
    if token != 'secret':
        await ws.close(code=1008)  # Policy violation
        return
    await ws.accept()
    await ws.send_text('Authentifie')
    await ws.close()

print('auth ws defini')

---

## 9. Rate limiting

In [ ]:
import time

class RateLimiter:
    def __init__(self, max_per_sec: int) -> None:
        self.max = max_per_sec
        self.history: list[float] = []

    def allow(self) -> bool:
        now = time.time()
        self.history = [t for t in self.history if now - t < 1.0]
        if len(self.history) >= self.max:
            return False
        self.history.append(now)
        return True

rl = RateLimiter(5)
print([rl.allow() for _ in range(7)])

---

## 10. Heartbeat et ping/pong

In [ ]:
# Le protocole WS a un mecanisme natif ping/pong
# Avec websockets : ping_interval=20 (defaut), ping_timeout=20
import websockets
# websockets.serve(handler, 'localhost', 8765, ping_interval=10, ping_timeout=5)
print('ping_interval et ping_timeout configurent le heartbeat')

---

## 11. Server-Sent Events (SSE)

**SSE** = flux unidirectionnel serveur -> client via HTTP. Plus simple que WS quand il n'y a pas besoin de bidirectionnel.

In [ ]:
from fastapi.responses import StreamingResponse
import asyncio

async def event_generator():
    for i in range(5):
        yield f'data: event {i}\n\n'
        await asyncio.sleep(0.1)

@api.get('/events')
async def events():
    return StreamingResponse(event_generator(), media_type='text/event-stream')

print('SSE defini')

---

## Synthese

| Element | FastAPI WebSocket |
|---|---|
| Endpoint | `@app.websocket('/path')` |
| Accept | `await ws.accept()` |
| Recevoir texte | `await ws.receive_text()` |
| Recevoir JSON | `await ws.receive_json()` |
| Envoyer texte | `await ws.send_text(...)` |
| Envoyer JSON | `await ws.send_json(...)` |
| Deconnexion | `WebSocketDisconnect` |
| Fermer | `await ws.close(code=1000)` |

### Regles a retenir

- **WS = bidirectionnel, connexion longue**. HTTP = requete-reponse.
- **Toujours gerer `WebSocketDisconnect`** pour nettoyer les ressources.
- **`ConnectionManager`** pour broadcast et lister les clients.
- **Heartbeat** pour detecter les clients morts (pings automatiques).
- **SSE** si unidirectionnel (notifications, flux).

---

## Exercices

### Exercice 1 (facile) -- Echo

Ecrivez un endpoint WebSocket FastAPI qui renvoie chaque message prefixe par `Echo: `.

In [ ]:
from fastapi import FastAPI, WebSocket

app = FastAPI()

# TODO

<details>
<summary>Voir la correction</summary>

```python
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
app = FastAPI()

@app.websocket('/echo')
async def echo(ws: WebSocket):
    await ws.accept()
    try:
        while True:
            msg = await ws.receive_text()
            await ws.send_text(f'Echo: {msg}')
    except WebSocketDisconnect:
        pass
```

</details>

### Exercice 2 (facile) -- Compteur serveur

Endpoint WebSocket qui envoie toutes les 200ms un compteur croissant (0, 1, 2, ...) pendant 5 secondes.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import asyncio
from fastapi import FastAPI, WebSocket
app = FastAPI()

@app.websocket('/counter')
async def counter(ws: WebSocket):
    await ws.accept()
    for i in range(25):
        await ws.send_text(str(i))
        await asyncio.sleep(0.2)
    await ws.close()
```

</details>

### Exercice 3 (moyen) -- Chat multi-client

Implementez un chat simple : chaque message envoye est broadcast a tous les autres clients connectes.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
app = FastAPI()
clients: list[WebSocket] = []

@app.websocket('/chat')
async def chat(ws: WebSocket):
    await ws.accept()
    clients.append(ws)
    try:
        while True:
            msg = await ws.receive_text()
            for c in clients:
                if c is not ws:
                    await c.send_text(msg)
    except WebSocketDisconnect:
        clients.remove(ws)
```

</details>

### Exercice 4 (moyen) -- Echange JSON structure

Endpoint WebSocket qui attend des messages JSON `{"op": "add", "a": 1, "b": 2}` et renvoie `{"result": 3}`. Gere `add`, `sub`, `mul`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
app = FastAPI()

@app.websocket('/calc')
async def calc(ws: WebSocket):
    await ws.accept()
    try:
        while True:
            data = await ws.receive_json()
            op = data.get('op')
            a, b = data.get('a', 0), data.get('b', 0)
            if op == 'add':
                result = a + b
            elif op == 'sub':
                result = a - b
            elif op == 'mul':
                result = a * b
            else:
                await ws.send_json({'error': 'unknown op'})
                continue
            await ws.send_json({'result': result})
    except WebSocketDisconnect:
        pass
```

</details>

### Exercice 5 (difficile) -- Chat authentifie avec pseudos

Variante de l'exercice 3 : le client doit d'abord envoyer `{"type": "hello", "name": "Alice"}`, puis tous les messages suivants sont broadcast avec `{"from": "Alice", "msg": "..."}`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
app = FastAPI()
clients: dict[WebSocket, str] = {}

@app.websocket('/chat-auth')
async def chat_auth(ws: WebSocket):
    await ws.accept()
    hello = await ws.receive_json()
    if hello.get('type') != 'hello' or 'name' not in hello:
        await ws.close(code=1008)
        return
    name = hello['name']
    clients[ws] = name
    try:
        while True:
            msg = await ws.receive_json()
            out = {'from': name, 'msg': msg.get('msg', '')}
            for c in clients:
                await c.send_json(out)
    except WebSocketDisconnect:
        clients.pop(ws, None)
```

</details>

---

## Ressources externes

- [websockets (bibliotheque)](https://websockets.readthedocs.io/)
- [FastAPI WebSockets](https://fastapi.tiangolo.com/advanced/websockets/)
- [RFC 6455 - The WebSocket Protocol](https://datatracker.ietf.org/doc/html/rfc6455)